# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIRˆ2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset adheres to the Croissant schema and its metadata can be found at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset's metadata
dataset = mlc.Dataset(croissant_url)
# Access .metadata as an object (not a dict!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, their IDs, and fields.

In [ ]:
# List all record sets by their '@id'.
print('Available record sets:')
record_set_ids = [r['@id'] for r in dataset.metadata.to_json().get('recordSet', [])]
if record_set_ids:
    for rs in dataset.metadata.to_json().get('recordSet', []):
        print(f"@id: {rs['@id']}\tname: {rs.get('name', '')}")
else:
    print('No record sets found in top-level metadata. Attempting to infer from distribution.')
    dist = dataset.metadata.to_json().get('distribution', [])
    for d in dist:
        print(f"Distribution: {d['@id']}")

# Try to get at least one records set ID for further exploration
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
else:
    # Fallback: Use first distribution '@id' if no explicit 'recordSet'.
    distributions = dataset.metadata.to_json().get('distribution', [])
    if distributions:
        primary_record_set_id = distributions[0]['@id']
    else:
        primary_record_set_id = None
print(f"\nPrimary record set ID for demo: {primary_record_set_id}")

### Fields/Columns in Record Set
We list an example of records (one per row) for the detected record set. All entities are referenced by their `@id`.


In [ ]:
print('First examples of records in the record set:')
if primary_record_set_id:
    try:
        for idx, rec in enumerate(dataset.records(record_set=primary_record_set_id)):
            print(rec)
            if idx > 2:
                break
    except Exception as e:
        print(f"Could not fetch records for record set {primary_record_set_id}: {e}")
else:
    print('No available record set to sample.')

## 3. Data Extraction
Load data from all available record sets into DataFrames.

For this dataset, we demonstrate loading with the detected record set IDs and print their fields (columns) for initial exploration.

In [ ]:
# Collect record set IDs for extraction
record_sets = record_set_ids
dataframes = {}

# If no explicit recordSet found, fall back on distribution IDs
if not record_sets:
    distributions = dataset.metadata.to_json().get('distribution', [])
    record_sets = [d['@id'] for d in distributions]

# Try to extract each record set
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f'Record set {rs_id}: columns = {dataframes[rs_id].columns.tolist()}')
        else:
            print(f'Record set {rs_id} returned no records.')
    except Exception as e:
        print(f'Error extracting records for {rs_id}: {e}')

# Pick the first valid dataframe for demo
df_key = next(iter(dataframes)) if dataframes else None
if df_key:
    display(dataframes[df_key].head())
else:
    print('No DataFrame extracted.')

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA steps. This includes filtering, normalization, and grouping. For demonstration, we select a numeric field by its id if available and a group field if appropriate.

In [ ]:
# Attempt EDA on the first non-empty dataframe
if df_key:
    df = dataframes[df_key]
    # Heuristically select the first numeric field
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f'Using numeric field: {numeric_field}')
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try group by another suitable field
        group_candidates = [c for c in df.columns if c != numeric_field and (df[c].dtype == object or df[c].dtype.name == 'category')]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
    else:
        print('No numeric field found for EDA.')
else:
    print('No available data for EDA.')

## 5. Visualization
Visualize the distribution of a numeric field or relationships between fields. We'll plot the distribution if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_key and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[df_key][numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field} (@id: {numeric_field})')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and perform basic analysis and visualization on a Croissant-schema dataset using `mlcroissant`. All references to record sets and columns use their `@id` for reproducibility. You can adapt these steps for further, domain-specific analysis.
